# Streaming and Kafka

Notebook for **Chapter 11** of the *de* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/de/ch11.html

Tip: change a number, predict the output, then run the cell again.

## 11.1 Event Time, Windows and Watermarks

In [ ]:
from collections import defaultdict
ev=[5,20,62,75,130,135,400]
def tumbling(ev,w=60):
    d=defaultdict(int)
    for t in ev: d[(t//w)*w]+=1
    return dict(sorted(d.items()))
def sliding(ev,w=60,s=30):
    d={}
    for st in range(0,max(ev)+1,s):
        c=sum(1 for t in ev if st<=t<st+w)
        if c: d[st]=c
    return d
def session(ev,gap=60):
    ss=[[ev[0]]]
    for t in ev[1:]:
        if t-ss[-1][-1]<=gap: ss[-1].append(t)
        else: ss.append([t])
    return [(s[0],s[-1],len(s)) for s in ss]
print("tumbling 60:",tumbling(ev)); print("sliding 60/30:",sliding(ev)); print("session gap 60:",session(ev))

In [ ]:
"""Event-time tumbling window (1 min) with watermark (allowed lateness 2 min)."""
from collections import defaultdict
events=[(5,"a"),(20,"b"),(62,"c"),(58,"d"),(130,"e"),(75,"f"),(10,"g"),(200,"h"),(125,"i"),(3,"j")]
W=60; LATE=120
counts=defaultdict(int); max_et=0; emitted={}; dropped=[]
def wstart(t): return (t//W)*W
for et,name in events:
    max_et=max(max_et,et); wm=max_et-LATE
    if wstart(et)+W<=wm:         # window already closed by watermark
        dropped.append((name,et)); print(f"event {name} t={et:3d} wm={wm:4d} -> DROPPED (too late)"); continue
    counts[wstart(et)]+=1
    print(f"event {name} t={et:3d} wm={wm:4d} -> window [{wstart(et)},{wstart(et)+W}) count={counts[wstart(et)]}")
    for w in sorted(list(counts)):
        if w+W<=wm and w not in emitted: emitted[w]=counts[w]; print(f"   emit final window [{w},{w+W}) = {counts[w]}")
print("final counts:",dict(sorted(counts.items())),"dropped:",dropped)

## 11.2 Kafka: The Log, Partitions, Consumer Groups and Delivery Semantics

In [ ]:
import zlib
P=3
def part(key): return zlib.crc32(key.encode())%P
msgs=[("u1","click"),("u2","buy"),("u1","view"),("u3","click"),("u2","view"),("u1","buy"),("u3","buy")]
logs={0:[],1:[],2:[]}
for k,v in msgs: logs[part(k)].append((k,v))
for p in range(P): print("partition",p,logs[p])

In [ ]:
def assign(consumers,parts):
    a={}
    for i,p in enumerate(parts): a.setdefault(consumers[i%len(consumers)],[]).append(p)
    return a
for n in (1,2,3,4):
    print(n,"consumers ->",assign([f"c{i+1}" for i in range(n)],[0,1,2]))

1 consumers -> {'c1': [0, 1, 2]}
2 consumers -> {'c1': [0, 2], 'c2': [1]}
3 consumers -> {'c1': [0], 'c2': [1], 'c3': [2]}
4 consumers -> {'c1': [0], 'c2': [1], 'c3': [2]}


In [ ]:
def run(commit_first):
    log=list(range(1,7)); committed=0; processed=[]; crashed=False
    while committed<len(log):
        off=committed
        if commit_first:
            committed=off+1
            if off==3 and not crashed: crashed=True; continue   # crash: message 4 lost
            processed.append(log[off])
        else:
            processed.append(log[off])
            if off==3 and not crashed: crashed=True; continue   # crash before commit: reprocess
            committed=off+1
    return processed
print("at-most-once:",run(True)); print("at-least-once:",run(False))
sink={}
for m in run(False): sink[m]=f"row{m}"
print("idempotent sink keys:",sorted(sink),"rows:",len(sink))

at-most-once: [1, 2, 3, 5, 6]
at-least-once: [1, 2, 3, 4, 4, 5, 6]
idempotent sink keys: [1, 2, 3, 4, 5, 6] rows: 6


In [ ]:
# not run here
from confluent_kafka import Consumer
c = Consumer({"bootstrap.servers": "localhost:9092",
              "group.id": "fraud",
              "enable.auto.commit": False,
              "auto.offset.reset": "earliest"})
c.subscribe(["orders"])
while True:
    msg = c.poll(1.0)
    if msg is None or msg.error():
        continue
    upsert_to_db(msg.key(), msg.value())   # idempotent write first
    c.commit(msg)                          # then commit: at-least-once

## 11.3 Processing Engines: Spark Structured Streaming, Flink and Real-Time OLAP

In [ ]:
# not run here
from pyspark.sql import functions as F
orders = (spark.readStream.format("kafka")
          .option("kafka.bootstrap.servers", "broker:9092")
          .option("subscribe", "orders").load()
          .select(F.from_json(F.col("value").cast("string"), schema).alias("o"))
          .select("o.*"))                         # columns: order_ts, city, amount

per_minute = (orders
    .withWatermark("order_ts", "2 minutes")       # allowed lateness
    .groupBy(F.window("order_ts", "1 minute"), "city")
    .agg(F.count("*").alias("orders"), F.sum("amount").alias("revenue")))

(per_minute.writeStream
    .outputMode("append")                         # emit a window once it is final
    .format("delta").option("checkpointLocation", "/chk/orders_per_min")
    .start("/lake/gold/orders_per_min"))

## 11.4 The End-to-End Exactly-Once Recipe, Failure Injection and Stream Joins

Everyone says "we need exactly-once". Experts ask: exactly once *where*? Section 11.2 showed that delivery between systems is at-least-once or at-most-once, and that "exactly once" really means **exactly-once effect**: the result looks as if each message was processed once, even though some messages were delivered twice. This section builds that, tests it by crashing the pipeline on purpose, and then covers joins with state.

In [ ]:
# Exactly-once effect = at-least-once delivery + idempotent or transactional sink.
# Pipeline: read from a log (offsets 0..5), add 10 to each value, write to a sink table.
import sqlite3
LOG = [(0, "k1", 5), (1, "k2", 7), (2, "k1", 3), (3, "k3", 1), (4, "k2", 2), (5, "k1", 4)]

def run(mode, crash_at):
    """crash_at: which step of the processing of message offset 3 fails (None = no crash)."""
    db = sqlite3.connect(":memory:")
    db.executescript("""CREATE TABLE sink(offset INTEGER, k TEXT, v INTEGER);
                        CREATE TABLE committed(id INTEGER PRIMARY KEY CHECK(id=1), next_offset INTEGER);
                        INSERT INTO committed VALUES (1, 0);""")
    crashed = False
    def process_from(start):
        nonlocal crashed
        for off, k, v in LOG[start:]:
            if mode == "naive-commit-first":
                db.execute("UPDATE committed SET next_offset=?", (off + 1,)); db.commit()       # commit offset, then write
                if off == 3 and crash_at == "after_offset_commit" and not crashed:
                    crashed = True; raise RuntimeError("crash")
                db.execute("INSERT INTO sink VALUES (?,?,?)", (off, k, v + 10)); db.commit()
            elif mode == "naive-write-first":
                db.execute("INSERT INTO sink VALUES (?,?,?)", (off, k, v + 10)); db.commit()    # write, then commit offset
                if off == 3 and crash_at == "after_write" and not crashed:
                    crashed = True; raise RuntimeError("crash")
                db.execute("UPDATE committed SET next_offset=?", (off + 1,)); db.commit()
            elif mode == "transactional":
                with db:                                                                         # one atomic transaction
                    db.execute("INSERT INTO sink VALUES (?,?,?)", (off, k, v + 10))
                    if off == 3 and crash_at == "mid_transaction" and not crashed:
                        crashed = True; raise RuntimeError("crash")
                    db.execute("UPDATE committed SET next_offset=?", (off + 1,))
            elif mode == "idempotent":
                db.execute("INSERT INTO sink SELECT ?,?,? WHERE NOT EXISTS (SELECT 1 FROM sink WHERE offset=?)", (off, k, v + 10, off)); db.commit()
                if off == 3 and crash_at == "after_write" and not crashed:
                    crashed = True; raise RuntimeError("crash")
                db.execute("UPDATE committed SET next_offset=?", (off + 1,)); db.commit()
    for attempt in range(2):                      # run, crash, restart from the committed offset
        try:
            start = db.execute("SELECT next_offset FROM committed").fetchone()[0]
            process_from(start); break
        except RuntimeError:
            pass
    rows = db.execute("SELECT offset FROM sink ORDER BY offset").fetchall()
    return len(rows), len(set(r[0] for r in rows))

print("mode                 crash point           rows  distinct  verdict")
for mode, crash in [("naive-commit-first", "after_offset_commit"), ("naive-write-first", "after_write"),
                    ("transactional", "mid_transaction"), ("idempotent", "after_write")]:
    n, d = run(mode, crash)
    verdict = "lost" if d < 6 else ("duplicates" if n > d else "exactly once")
    print(f"{mode:20s} {crash:20s} {n:5d} {d:9d}  {verdict}")

mode                 crash point           rows  distinct  verdict
naive-commit-first   after_offset_commit      5         5  lost
naive-write-first    after_write              7         6  duplicates
transactional        mid_transaction          6         6  exactly once
idempotent           after_write              6         6  exactly once


In [ ]:
# not run here (needs a Kafka broker and the confluent-kafka package)
from confluent_kafka import Consumer, Producer

consumer = Consumer({"bootstrap.servers": "localhost:9092", "group.id": "enricher",
                     "enable.auto.commit": False, "isolation.level": "read_committed",
                     "auto.offset.reset": "earliest"})
producer = Producer({"bootstrap.servers": "localhost:9092", "enable.idempotence": True,
                     "transactional.id": "enricher-1"})        # one id per producer instance
consumer.subscribe(["orders"])
producer.init_transactions()
while True:
    msg = consumer.poll(1.0)
    if msg is None or msg.error():
        continue
    producer.begin_transaction()
    producer.produce("orders_clean", key=msg.key(), value=msg.value().upper())
    producer.send_offsets_to_transaction(consumer.position(consumer.assignment()),
                                         consumer.consumer_group_metadata())
    producer.commit_transaction()      # output and consumed offsets become visible together

In [ ]:
# not run here (needs Spark and Delta Lake)
from delta.tables import DeltaTable

def upsert_batch(batch_df, batch_id):
    target = DeltaTable.forName(spark, "silver.orders")
    (target.alias("t")
        .merge(batch_df.alias("s"), "t.order_id = s.order_id")   # key comes from the data
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())                                              # same batch twice gives the same table

(orders_stream.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", "/chk/silver_orders")          # offsets and progress live here
    .start())

In [ ]:
# Stream-stream join with state TTL: match click to impression within 10 minutes; drop state after that.
impr = [("a", 0), ("b", 100), ("c", 200)]                 # (ad id, event time in seconds)
clicks = [("a", 120), ("b", 900), ("z", 50)]
TTL = 600
state, matched, late = {}, [], []
events = sorted([("impr", k, t) for k, t in impr] + [("click", k, t) for k, t in clicks], key=lambda e: e[2])
peak = 0
for kind, k, t in events:
    # evict state older than TTL relative to the current event time (watermark = event time here)
    state = {key: ts for key, ts in state.items() if t - ts <= TTL}
    if kind == "impr":
        state[k] = t
    else:
        if k in state: matched.append((k, t - state[k]))
        else: late.append((k, t))
    peak = max(peak, len(state))
print("matched (ad, seconds after impression):", matched)
print("unmatched clicks (no impression within TTL):", late)
print("peak state size:", peak, "keys")

matched (ad, seconds after impression): [('a', 120)]
unmatched clicks (no impression within TTL): [('z', 50), ('b', 900)]
peak state size: 3 keys


## Chapter recap

## Practice set

**1.** (easy) Tumbling windows of 5 minutes. Which window holds an event at 14:07:30?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/de/ch11.html